<a href="https://colab.research.google.com/github/kevin231/mycar/blob/main/simple_lora_trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
# 1. 進入 sd-scripts 資料夾
cd /content/sd-scripts

# 2. 強制定義 Python 搜尋路徑包含當前目錄 (.) 以及完整路徑
export PYTHONPATH=$PYTHONPATH:/content/sd-scripts:.
export LD_PRELOAD=""

In [ ]:

#@title Train LoRA
from google.colab import drive
drive.mount('/content/drive')
import os
# 強制清空會導致當機的環境變數
os.environ.pop('LD_PRELOAD', None)
print("✅ 已停用 tcmalloc，解決 Invalid Pointer 崩潰問題")

!apt -y update -qq
!wget http://launchpadlibrarian.net/367274644/libgoogle-perftools-dev_2.5-2.2ubuntu3_amd64.deb
!wget https://launchpad.net/ubuntu/+source/google-perftools/2.5-2.2ubuntu3/+build/14795286/+files/google-perftools_2.5-2.2ubuntu3_all.deb
!wget https://launchpad.net/ubuntu/+source/google-perftools/2.5-2.2ubuntu3/+build/14795286/+files/libtcmalloc-minimal4_2.5-2.2ubuntu3_amd64.deb
!wget https://launchpad.net/ubuntu/+source/google-perftools/2.5-2.2ubuntu3/+build/14795286/+files/libgoogle-perftools4_2.5-2.2ubuntu3_amd64.deb
!apt install -qq libunwind8-dev
!dpkg -i *.deb
%env LD_PRELOAD=libtcmalloc.so
!rm *.deb

!apt -y install -qq aria2
#!pip install torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 torchtext==0.14.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu116 -U
#!pip install -q xformers==0.0.16 triton==2.0.0 -U

!git clone https://github.com/camenduru/sd-scripts
!sed -i -e 's/requests==2.28.2/# requests==2.28.2/' /content/sd-scripts/requirements.txt
%cd /content/sd-scripts
#!pip install -q -r requirements.txt

model_url = "https://civitai.com/models/23900/anylora-checkpoint" #@param {type:"string"}
model_name = "anyloraCheckpoint_bakedvaeBlessedFp16.safetensors" #@param {type:"string"}
pretrained_model_name_or_path = f"/content/model/{model_name}"

!mkdir /content/model
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{model_url}' -d /content/model -o {model_name}
print('路徑設置')
root_path = "/content/drive/MyDrive/data" #@param {type:"string"}
instance_prompt = "images-50" #@param {type:"string"}
images_path = f"{root_path}/{instance_prompt}"
images_tags_json = f"{images_path}/{instance_prompt}.json"
images_latents_json = f"{images_path}/{instance_prompt}-latents.json"
output_dir = f"/content/drive/MyDrive/data/{instance_prompt}"
max_train_steps = 1600 #@param {type:"integer"}
print('開始訓練')
!pip install tensorflow>=2.16.0

!python -u finetune/tag_images_by_wd14_tagger.py {images_path} --repo_id SmilingWolf/wd-v1-4-convnext-tagger-v2 --model_dir wd14_tagger_model --thresh 0.35 --batch_size 1 --caption_extension .txt
!python -u finetune/merge_dd_tags_to_metadata.py {images_path} {images_tags_json} --caption_extension .txt
!python -u finetune/prepare_buckets_latents.py {images_path} {images_tags_json} {images_latents_json} {pretrained_model_name_or_path} --batch_size 1 --max_resolution 512,512 --min_bucket_reso 256 --max_bucket_reso 1024 --bucket_reso_steps 64 --mixed_precision no
!python -u train_network.py --pretrained_model_name_or_path {pretrained_model_name_or_path} --train_data_dir {images_path} --in_json {images_latents_json} --output_dir {output_dir} --xformers --max_train_steps {max_train_steps} --use_8bit_adam --network_module networks.lora

!mv {output_dir}/last.safetensors {output_dir}/{instance_prompt}.safetensors

In [ ]:
import os

# 定義需要修正的檔案路徑
target_file = "/content/sd-scripts/library/train_util.py"

if os.path.exists(target_file):
    with open(target_file, 'r') as f:
        content = f.read()

    # 執行關鍵替換：將舊路徑改為新版 diffusers 的路徑
    # 將 diffusers.models.unet_2d_condition 修正為 diffusers.models.unets.unet_2d_condition
    updated_content = content.replace(
        "diffusers.models.unet_2d_condition",
        "diffusers.models.unets.unet_2d_condition"
    )

    with open(target_file, 'w') as f:
        f.write(updated_content)
    print("✅ 成功！已將 UNet 呼叫路徑修正為新版規格。")
else:
    print("❌ 找不到檔案，請確認 /content/sd-scripts/library/train_util.py 是否存在。")

In [ ]:
%%bash
# 1. 強制重置環境變數，預防記憶體錯誤
export LD_PRELOAD=""
cd /content/sd-scripts
export PYTHONPATH=$PYTHONPATH:$(pwd)

# 2. 確認訓練參數 (請檢查你的模型檔名是否為 anylora... .safetensors)
MODEL_PATH="/content/model/anyloraCheckpoint_bakedvaeBlessedFp16.safetensors"
TRAIN_DATA="/content/drive/MyDrive/data/images-50"
OUTPUT_DIR="/content/drive/MyDrive/data/images-50/output"

mkdir -p $OUTPUT_DIR

# 3. 執行正式訓練指令
python train_network.py \
  --pretrained_model_name_or_path=$MODEL_PATH \
  --train_data_dir=$TRAIN_DATA \
  --output_dir=$OUTPUT_DIR \
  --instance_prompt="sk_autodrive" \
  --resolution=512 \
  --mixed_precision="fp16" \
  --train_batch_size=1 \
  --learning_rate=1e-4 \
  --max_train_steps=1600 \
  --save_every_n_epochs=1 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --xformers \
  --lr_scheduler="cosine" \
  --gradient_checkpointing

In [ ]:
%%bash
# 1. 環境與路徑初始化
export LD_PRELOAD=""
cd /content/sd-scripts
export PYTHONPATH=$PYTHONPATH:.

# 2. 強力修正原始碼 (確保 logging_dir 徹底移除)
sed -i 's/logging_dir=args.logging_dir,//g' /content/sd-scripts/library/train_util.py
echo "✅ 原始碼修正檢查完成"

# 3. 設定訓練路徑
MODEL_PATH="/content/model/anyloraCheckpoint_bakedvaeBlessedFp16.safetensors"
TRAIN_DATA="/content/drive/MyDrive/data/images-50"
OUTPUT_DIR="/content/drive/MyDrive/data/images-50/output"

mkdir -p "$OUTPUT_DIR"

# 4. 執行正式訓練 (確保反斜線後直接換行，沒有空格或註解)
python train_network.py \
  --pretrained_model_name_or_path="$MODEL_PATH" \
  --train_data_dir="$TRAIN_DATA" \
  --output_dir="$OUTPUT_DIR" \
  --resolution=512 \
  --mixed_precision="fp16" \
  --train_batch_size=1 \
  --learning_rate=1e-4 \
  --max_train_steps=1600 \
  --save_every_n_epochs=1 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --xformers \
  --lr_scheduler="cosine" \
  --gradient_checkpointing

In [ ]:
%%bash
# 1. 環境與路徑初始化
cd /content/sd-scripts
export PYTHONPATH=$PYTHONPATH:.

# 2. 強制修正原始碼 (移除 API 衝突)
sed -i 's/logging_dir=args.logging_dir,//g' library/train_util.py

# 3. 解決 tcmalloc 記憶體衝突的環境變數
export MALLOC_CONF="dirty_decay_ms:0,muzzy_decay_ms:0"

# 4. 定義路徑
MODEL="/content/model/anyloraCheckpoint_bakedvaeBlessedFp16.safetensors"
DATA="/content/drive/MyDrive/data/images-50"
OUT="/content/drive/MyDrive/data/images-50/output"

mkdir -p "$OUT"

# 5. 正式啟動訓練 (移除 --xformers，改用內建優化)
python train_network.py \print('開始訓練')
  --pretrained_model_name_or_path="$MODEL" \
  --train_data_dir="$DATA" \
  --output_dir="$OUT" \
  --resolution=512 \
  --mixed_precision="fp16" \
  --train_batch_size=1 \
  --learning_rate=1e-4 \
  --max_train_steps=1600 \
  --save_every_n_epochs=1 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --lr_scheduler="cosine" \
  --gradient_checkpointing print('訓練結束')

In [ ]:
%%bash
# 1. 環境與路徑初始化
export LD_PRELOAD=""
cd /content/sd-scripts
export PYTHONPATH=$PYTHONPATH:.

# 2. 強力修正原始碼 (確保 logging_dir 徹底移除)
sed -i 's/logging_dir=args.logging_dir,//g' /content/sd-scripts/library/train_util.py
echo "✅ 原始碼修正檢查完成"

# 3. 設定訓練路徑
MODEL_PATH="/content/model/anyloraCheckpoint_bakedvaeBlessedFp16.safetensors"
TRAIN_DATA="/content/drive/MyDrive/data/images-50"
OUTPUT_DIR="/content/drive/MyDrive/data/images-50/output"

mkdir -p "$OUTPUT_DIR"

# 4. 執行正式訓練 (確保反斜線後直接換行，沒有空格或註解)
python train_network.py \
  --pretrained_model_name_or_path="$MODEL_PATH" \
  --train_data_dir="$TRAIN_DATA" \
  --output_dir="$OUTPUT_DIR" \
  --resolution=512 \
  --mixed_precision="fp16" \
  --train_batch_size=1 \
  --learning_rate=1e-4 \
  --max_train_steps=1600 \
  --save_every_n_epochs=1 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --xformers \
  --lr_scheduler="cosine" \
  --gradient_checkpointing

In [ ]:
import os

file_path = "/content/sd-scripts/library/train_util.py"

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    with open(file_path, 'w') as f:
        for line in lines:
            # 找到含有 logging_dir=args.logging_dir 的那一行並移除該參數
            if "logging_dir=args.logging_dir," in line:
                line = line.replace("logging_dir=args.logging_dir,", "")
            f.write(line)

    print("✅ 成功：已修復 Accelerator 初始化參數衝突。")
else:
    print("❌ 找不到檔案，請確認 /content/sd-scripts/library/train_util.py 是否存在。")

In [ ]:
import os
import shutil

# 1. 定義搜尋起點
search_root = "/content/drive/MyDrive/data/images-50"

def fix_dataset_folder(root):
    found = False
    for item in os.listdir(root):
        item_path = os.path.join(root, item)
        # 如果找到資料夾且裡面有圖片 (jpg/png)
        if os.path.isdir(item_path):
            files = os.listdir(item_path)
            has_images = any(f.lower().endswith(('.jpg', '.png', '.jpeg')) for f in files)

            if has_images:
                # 檢查是否已經符合 "數字_標籤" 格式，如果不符合就改名
                if "_" not in item or not item.split("_")[0].isdigit():
                    new_name = os.path.join(root, "20_sk_autodrive")
                    # 如果目標已存在，先合併或刪除
                    if os.path.exists(new_name):
                        shutil.rmtree(new_name)
                    os.rename(item_path, new_name)
                    print(f"✅ 成功：已將【{item}】改名為【20_sk_autodrive】")
                    found = True
                    break
                else:
                    print(f"✨ 資料夾【{item}】格式已經正確，無需修改。")
                    found = True
                    break
    if not found:
        print("❌ 警告：在指定目錄下找不到任何包含圖片的子資料夾，請手動檢查雲端硬碟。")

fix_dataset_folder(search_root)

In [ ]:
import os

file_path = "/content/sd-scripts/library/train_util.py"

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 針對性修正：移除包含 logging_dir 的那一行參數
    # 我們把 "logging_dir=args.logging_dir," 替換為空字串
    fixed_content = content.replace("logging_dir=args.logging_dir,", "")

    # 有些版本可能沒逗號，保險起見再清一次
    fixed_content = fixed_content.replace("logging_dir=args.logging_dir", "")

    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(fixed_content)

    print("✅ 修正完成！已手動拔除 Accelerator 的 logging_dir 接腳。")
else:
    print("❌ 找不到檔案，請確認 /content/sd-scripts 是否存在。")